# River correction Workflow (Python)

Python workflow for sediment porosity processing and static inspection plots using NumPy, xarray, rasterio, and Matplotlib.

In [4]:
# georeference the bathymetry from GETM topo.nc and save as GeoTIFF for use in GIS or other mapping tools.

import numpy as np
from scipy.interpolate import griddata
from pathlib import Path

from datetime import datetime
import os
import pandas as pd
import xarray as xr

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
preproc_indir = Path('/export/lv9/user/cgiannopoulos/home/pre-processing/')
setup_indir = Path('/export/lv9/user/cgiannopoulos/home/GETM_ERSEM_SETUPS/dws_500m/')
base_dir = Path('/export/lv9/user/cgiannopoulos/home/pre-processing/rivers/')
topo_nc = Path('/export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_resampling/topo_dws_500m.nc')

rivers_info = setup_indir / 'riverinfo.dat'
output_file = preproc_indir / 'rivers_bfm_20260518_WaddenSea_all_Christos.nc'

print(f'Topo path:   {topo_nc}')
print(f'Final out:   {output_file}')

Topo path:   /export/lv9/user/cgiannopoulos/home/pre-processing/bathymetry_resampling/topo_dws_500m.nc
Final out:   /export/lv9/user/cgiannopoulos/home/pre-processing/rivers_bfm_20260518_WaddenSea_all_Christos.nc


In [7]:
import numpy as np
import pandas as pd
import xarray as xr

rivers_info = setup_indir / 'riverinfo.dat'
rivers_file = setup_indir / 'rivers.nc'
r14_file = preproc_indir / 'rivers/r14.txt'
r37_file = preproc_indir / 'rivers/r37.txt'
rivers_output_file = preproc_indir / 'rivers/rivers_bfm_20260518_WaddenSea_all_Christos.nc'

In [10]:
# Inspect source structures and replace r14/r37 with corrected time series from text files.

import re

def _read_discharge_series(txt_path):
    """Read a river discharge text file and return a 1D float NumPy array."""
    values = []
    with open(txt_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue

            # Split on whitespace, comma, or semicolon and keep numeric tokens.
            tokens = re.split(r"[\s,;]+", line)
            numeric = []
            for tok in tokens:
                tok = tok.strip().replace("D", "E").replace("d", "e")
                if tok == "":
                    continue
                try:
                    numeric.append(float(tok))
                except ValueError:
                    continue

            if numeric:
                # Use the last numeric field as discharge value.
                values.append(numeric[-1])

    series = np.asarray(values, dtype=float)
    if series.size == 0:
        raise ValueError(f"No numeric discharge values found in {txt_path}")
    return series

# Open and inspect rivers.nc
with xr.open_dataset(rivers_file) as ds_in:
    print("=== rivers.nc structure ===")
    print(ds_in)
    print("\nVariables:", list(ds_in.data_vars))
    print("Dimensions:", dict(ds_in.sizes))

    # Inspect r14.txt and r37.txt
    r14_values = _read_discharge_series(r14_file)
    r37_values = _read_discharge_series(r37_file)

    print("\n=== Text file summary ===")
    print(f"r14.txt values: {r14_values.size}, first 5: {r14_values[:5]}")
    print(f"r37.txt values: {r37_values.size}, first 5: {r37_values[:5]}")

    # Match length to model time axis
    time_dim = "time"
    if time_dim not in ds_in.dims:
        raise KeyError(f"Expected '{time_dim}' dimension in rivers.nc, found: {list(ds_in.dims)}")

    n_time = ds_in.sizes[time_dim]
    if r14_values.size != n_time or r37_values.size != n_time:
        raise ValueError(
            f"Length mismatch: time={n_time}, r14={r14_values.size}, r37={r37_values.size}. "
            "Ensure text files have exactly one value per model time step."
        )

    # Create editable copy and preserve metadata on replacement
    ds_out = ds_in.copy(deep=True)

    if "r14" not in ds_out.data_vars or "r37" not in ds_out.data_vars:
        raise KeyError("Expected variables 'r14' and 'r37' in rivers.nc")

    r14_attrs = ds_out["r14"].attrs
    r37_attrs = ds_out["r37"].attrs

    ds_out["r14"] = xr.DataArray(r14_values, dims=(time_dim,), coords={time_dim: ds_out[time_dim]})
    ds_out["r37"] = xr.DataArray(r37_values, dims=(time_dim,), coords={time_dim: ds_out[time_dim]})

    ds_out["r14"].attrs.update(r14_attrs)
    ds_out["r37"].attrs.update(r37_attrs)

    # Write corrected netCDF
    ds_out.to_netcdf(rivers_output_file)

print(f"Corrected file written to: {rivers_output_file}")

=== rivers.nc structure ===
<xarray.Dataset> Size: 57MB
Dimensions:    (nshort: 1, time: 23376)
Coordinates:
  * nshort     (nshort) int32 4B 1
  * time       (time) datetime64[ns] 187kB 1957-01-01 1957-01-02 ... 2020-12-31
Data variables: (12/368)
    limit_B1c  (nshort) float64 8B ...
    limit_B1n  (nshort) float64 8B ...
    limit_B1p  (nshort) float64 8B ...
    limit_Bac  (nshort) float64 8B ...
    limit_Ban  (nshort) float64 8B ...
    limit_Bap  (nshort) float64 8B ...
    ...         ...
    r77        (time) float64 187kB ...
    r77_N3n    (time) float64 187kB ...
    r77_N1p    (time) float64 187kB ...
    r78        (time) float64 187kB ...
    r78_N3n    (time) float64 187kB ...
    r78_N1p    (time) float64 187kB ...
Attributes:
    creator:               Sonja van Leeuwen: sonja.van.leeuwen@nioz.nl, NIOZ...
    title:                 European riverine input file
    comment:               Generated from different sources, see river histor...
    generation_date:       